In [22]:
import sys
import os
import glob

# ── Remove any stale pip_archive entries that break standalone numpy ────────
# (These get added if this cell is re-run after a failed previous session.)
sys.path = [p for p in sys.path if "pip_archive" not in p]
# Force numpy to reload from kit/python site-packages (numpy 2.x, the correct one)
for _mod in list(sys.modules.keys()):
    if _mod == "numpy" or _mod.startswith("numpy."):
        del sys.modules[_mod]

# When running under the "Isaac Sim Python 3" kernel, ISAAC_SIM_PATH,
# ISAAC_PATH, PYTHONPATH and LD_LIBRARY_PATH are already configured in
# kernel.json – nothing to add.
# When running under the default Python 3 kernel we attempt to find and
# configure Isaac Sim paths ourselves as a fallback.

_from_kernel = bool(os.environ.get("ISAAC_JUPYTER_KERNEL"))
isaac_sim_path = os.environ.get("ISAAC_SIM_PATH", "")

if not _from_kernel or not isaac_sim_path:
    # Fallback path detection
    _candidates = [
        "/home/cvincen6/isaacsim",
        *sorted(glob.glob(os.path.expanduser("~/.local/share/ov/pkg/isaac-sim-*"))),
        *sorted(glob.glob(os.path.expanduser("~/isaac-sim*"))),
    ]
    isaac_sim_path = next(
        (p for p in _candidates if os.path.isdir(p) and os.path.exists(os.path.join(p, "python.sh"))),
        None
    )
    if isaac_sim_path:
        _s = isaac_sim_path
        for p in reversed([
            f"{_s}/kit/python/lib/python3.11",
            f"{_s}/kit/python/lib/python3.11/site-packages",
            f"{_s}/python_packages",
            f"{_s}/exts/isaacsim.simulation_app",
            f"{_s}/extsDeprecated/omni.isaac.kit",
            f"{_s}/kit/kernel/py",
            f"{_s}/kit/plugins/bindings-python",
            f"{_s}/exts/omni.isaac.core_archive/pip_prebundle",
            f"{_s}/exts/omni.isaac.ml_archive/pip_prebundle",
            f"{_s}/exts/omni.pip.compute/pip_prebundle",
            f"{_s}/exts/omni.pip.cloud/pip_prebundle",
        ]):
            if os.path.isdir(p) and p not in sys.path:
                sys.path.insert(0, p)
        os.environ.setdefault("ISAAC_SIM_PATH", isaac_sim_path)
        os.environ.setdefault("ISAAC_PATH", isaac_sim_path)
        os.environ.setdefault("ISAAC_JUPYTER_KERNEL", "1")

isaac_sim_candidates = [isaac_sim_path] if isaac_sim_path else []  # compatibility

# ── Ensure vars required by SimulationApp.__init__ are always set ──────────
# CARB_APP_PATH and EXP_PATH are now in kernel.json, but set them as a
# fallback for non-kernel environments too.
if isaac_sim_path:
    os.environ.setdefault("CARB_APP_PATH", os.path.join(isaac_sim_path, "kit"))
    os.environ.setdefault("EXP_PATH",      os.path.join(isaac_sim_path, "apps"))
    os.environ.setdefault("ISAAC_PATH",    isaac_sim_path)
    os.environ.setdefault("ISAAC_SIM_PATH",isaac_sim_path)

import numpy as _np_ver_check
if isaac_sim_path:
    print(f"✓ Isaac Sim detected: {isaac_sim_path}")
    print(f"  Kernel-configured: {_from_kernel}")
else:
    print("✗ Isaac Sim not found.")
    print("  ➜ Switch kernel to 'Isaac Sim Python 3' (kernel selector, top-right)")
    print("  ➜ Or install NVIDIA Isaac Sim 4.0+ to /home/<user>/isaacsim")

print(f"✓ Python executable: {sys.executable}")
print(f"✓ Python version: {sys.version}")
print(f"✓ NumPy version: {_np_ver_check.__version__} ({_np_ver_check.__file__})")
print(f"✓ CARB_APP_PATH: {os.environ.get('CARB_APP_PATH', 'NOT SET')}")
print(f"✓ EXP_PATH: {os.environ.get('EXP_PATH', 'NOT SET')}")


# Franka Pick-and-Place with RL - Isaac Sim Edition

Training a Franka Emika Panda robot to pick colored boxes using reinforcement learning (PPO) in NVIDIA Isaac Sim.

## Setup and Imports

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
import time
import torch
from datetime import timedelta
import os

# Isaac Sim 4.x: import SimulationApp directly from the top-level isaacsim package.
# isaacsim bootstraps the Omniverse runtime; SimulationApp MUST be created
# before any other omni.* imports.
try:
    from isaacsim import SimulationApp
    print("✓ Using Isaac Sim 4.x API  (from isaacsim import SimulationApp)")
except ModuleNotFoundError:
    # Legacy fallback for Isaac Sim < 4.0
    from omni.isaac.kit import SimulationApp  # type: ignore
    print("✓ Using legacy Isaac Sim API (omni.isaac.kit)")

_isaac_sim_path = os.environ.get("ISAAC_SIM_PATH", "")

# EXP_PATH must be set so that SimulationApp can locate kit experience files
# when no explicit 'experience' key is provided.
_apps_dir = os.path.join(_isaac_sim_path, "apps")
if os.path.isdir(_apps_dir):
    os.environ.setdefault("EXP_PATH", _apps_dir)

# Prefer the headless Python experience; fall back to the base Python kit.
_experience_candidates = [
    os.path.join(_apps_dir, "omni.isaac.sim.python.gym.headless.rendering.kit"),
    os.path.join(_apps_dir, "omni.isaac.sim.python.headless.kit"),
    os.path.join(_apps_dir, "isaacsim.exp.base.python.kit"),
    os.path.join(_apps_dir, "isaacsim.exp.base.kit"),
]
_experience = next((p for p in _experience_candidates if os.path.isfile(p)), "")

CONFIG = {"headless": True}
if _experience:
    CONFIG["experience"] = _experience
    print(f"✓ Using experience: {os.path.basename(_experience)}")
else:
    print("⚠ No matching experience kit found; SimulationApp will auto-select.")

try:
    simulation_app = SimulationApp(CONFIG)
    print("✓ Isaac Sim initialized")
except Exception as e:
    print(f"✗ Isaac Sim initialization failed: {e}")
    raise

import omni
from omni.isaac.core import World
from omni.isaac.core.robots import Robot
from omni.isaac.core.utils.types import ArticulationAction
from omni.isaac.core.utils.nucleus import get_assets_root_path
from omni.isaac.core.objects import DynamicCuboid

print(f"✓ PyTorch using: {'cuda' if torch.cuda.is_available() else 'cpu'} device")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ All imports successful")


## Environment: Isaac Sim Pick and Place

In [ ]:
class PickPlaceEnvIsaacSim(gym.Env):
    """
    Isaac Sim environment for Franka Panda to pick and place a target block.

    Observation (31D):
      [0:7]   - Arm joint positions (7 DOF)
      [7:14]  - Arm joint velocities (7 DOF)
      [14:17] - End-effector XYZ position
      [17:20] - Block XYZ position
      [20:23] - Block XYZ velocity
      [23]    - Gripper width (sum of finger positions)
      [24]    - Contact flag (end-effector near block)
      [25]    - Reaching distance milestone
      [26]    - Grasping milestone
      [27]    - Lifting milestone
      [28]    - Sorting distance milestone
      [29:31] - Padding for compatibility

    Action (8D):
      [0:7]   - Arm joint position deltas (normalized -1 to 1)
      [7]     - Gripper command: -1=open, +1=close
    """

    def __init__(self, headless=True):
        super().__init__()

        # Create Isaac Sim world
        self.world = World(
            stage_units_in_meters=1.0,
            backend="torch",
            device="cuda" if torch.cuda.is_available() else "cpu",
        )

        # --- Load Franka from built-in Isaac Sim asset library ---
        assets_root = get_assets_root_path()
        if assets_root is None:
            raise RuntimeError(
                "Could not reach Isaac Sim Nucleus asset server. "
                "Make sure Isaac Sim is properly installed and Nucleus is running."
            )
        franka_usd = assets_root + "/Isaac/Robots/Franka/franka.usd"
        try:
            self.franka = self.world.scene.add(
                Robot(
                    prim_path="/World/Franka",
                    name="franka",
                    usd_path=franka_usd,
                )
            )
            print(f"✓ Franka robot loaded from: {franka_usd}")
        except Exception as e:
            print(f"✗ Could not load Franka USD: {e}")
            self.franka = None

        # --- Add a real rigid-body cube as the target block ---
        # DynamicCuboid has full physics (mass, contacts, velocity) unlike XFormPrim
        try:
            self.target_block = self.world.scene.add(
                DynamicCuboid(
                    prim_path="/World/target_block",
                    name="target_block",
                    position=np.array([0.5, 0.0, 0.035]),
                    scale=np.array([0.04, 0.04, 0.04]),
                    color=np.array([1.0, 0.0, 0.0]),  # red
                    mass=0.1,
                )
            )
            print("✓ Target block added (DynamicCuboid with physics)")
        except Exception as e:
            print(f"✗ Could not add target block: {e}")
            self.target_block = None

        # Episode configuration
        self.max_episode_steps = 300
        self.step_count = 0

        # Gymnasium spaces (same as MuJoCo version for compatibility)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(31,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(8,), dtype=np.float32)

        # Control parameters
        self.max_action_step = 0.05

    def reset(self, seed=None):
        """Reset environment to initial state."""
        super().reset(seed=seed)
        self.step_count = 0

        try:
            self.world.reset()

            if self.franka is not None:
                neutral_pose = np.array([0.0, 0.5, 0.0, -1.5, 0.0, 1.5, 0.785])
                self.franka.set_joint_positions(neutral_pose[:7])
                self.franka.set_joint_positions(np.array([0.04, 0.04]), joint_indices=np.array([7, 8]))

            if self.target_block is not None:
                self.target_block.set_world_pose(position=np.array([0.5, 0.0, 0.035]))
                self.target_block.set_linear_velocity(np.array([0.0, 0.0, 0.0]))
                self.target_block.set_angular_velocity(np.array([0.0, 0.0, 0.0]))

            # Let the scene settle
            for _ in range(100):
                self.world.step(render=False)

        except Exception as e:
            print(f"⚠ Reset warning: {e}")

        return self._get_obs(), {}

    def _get_obs(self):
        """Get observation as 31D vector."""
        obs = np.zeros(31, dtype=np.float32)

        try:
            if self.franka is not None:
                arm_pos = self.franka.get_joint_positions()[:7]
                arm_vel = self.franka.get_joint_velocities()[:7]
                obs[0:7] = arm_pos
                obs[7:14] = arm_vel

                # End-effector position (Franka hand link index 10)
                ee_pos = self.franka.get_link_positions(link_indices=np.array([10]))[0]
                obs[14:17] = ee_pos

                # Gripper width
                gripper_q = self.franka.get_joint_positions()[7:9]
                obs[23] = float(np.sum(gripper_q))

            if self.target_block is not None:
                block_pos, _ = self.target_block.get_world_pose()
                # DynamicCuboid exposes a real linear velocity
                block_vel = self.target_block.get_linear_velocity()
                obs[17:20] = block_pos
                obs[20:23] = block_vel

        except Exception as e:
            print(f"⚠ Observation warning: {e}")

        return obs

    def step(self, action):
        """Execute action and return (obs, reward, terminated, truncated, info)."""
        action = np.clip(action, -1.0, 1.0)

        try:
            if self.franka is not None:
                current_q = self.franka.get_joint_positions()

                arm_deltas = action[0:7] * self.max_action_step
                target_q = current_q[:7] + arm_deltas
                target_q = np.clip(target_q, -2.8973, 2.8973)
                target_q[5] = -target_q[3]  # keep gripper parallel

                self.franka.apply_action(ArticulationAction(joint_positions=target_q))

                # Gripper: map [-1, 1] → [0.04, 0.0] m
                gripper_width = (1.0 - action[7]) / 2.0 * 0.04
                self.franka.set_joint_positions(
                    np.array([gripper_width, gripper_width]),
                    joint_indices=np.array([7, 8]),
                )

            self.world.step(render=False)

        except Exception as e:
            print(f"⚠ Step warning: {e}")

        self.step_count += 1
        obs = self._get_obs()

        # Reward
        reward = -0.001  # time penalty
        try:
            if self.franka is not None and self.target_block is not None:
                ee_pos = obs[14:17]
                block_pos = obs[17:20]
                horiz_dist = np.linalg.norm(ee_pos[0:2] - block_pos[0:2])
                vert_dist = ee_pos[2] - block_pos[2]
                ee_to_block = np.linalg.norm(ee_pos - block_pos)

                if horiz_dist < 0.15:
                    reward += 1.0
                if horiz_dist < 0.08 and abs(vert_dist - 0.05) < 0.03:
                    reward += 2.0
                if ee_to_block < 0.08:
                    reward += 5.0
                if block_pos[2] > 0.10:   # block lifted off table
                    reward += 10.0
        except Exception as e:
            print(f"⚠ Reward warning: {e}")

        terminated = self.step_count >= self.max_episode_steps

        return obs, float(reward), terminated, False, {}

    def close(self):
        """Clean up Isaac Sim world."""
        try:
            if self.world is not None:
                self.world.clear()
        except Exception as e:
            print(f"⚠ Close warning: {e}")


print("✓ PickPlaceEnvIsaacSim class defined")


In [ ]:
def train_pick_place_isaac(total_timesteps=100000):
    """Train Franka Panda gripper to pick up a single block using Isaac Sim."""
    print("=" * 70)
    print("Training: Pick up a single block (Isaac Sim)")
    print("=" * 70)
    print(f"Task: Train gripper to reach and pick up one target block")
    print(f"Physics Engine: Isaac Sim (NVIDIA ODE)")
    print(f"Device: {('CUDA' if torch.cuda.is_available() else 'CPU')}") 
    print()
    
    # Create vectorized environments
    def make_env():
        return PickPlaceEnvIsaacSim(headless=True)
    
    try:
        env = make_vec_env(make_env, n_envs=1)  # Start with 1 env, can scale to 4+
        
        # Train with PPO
        model = PPO(
            "MlpPolicy",
            env,
            learning_rate=3e-4,
            n_steps=1024,
            batch_size=64,
            n_epochs=20,
            gamma=0.99,
            gae_lambda=0.95,
            clip_range=0.2,
            verbose=1,
            policy_kwargs={"net_arch": [256, 256]},
            device="cuda" if torch.cuda.is_available() else "cpu"
        )
        
        print(f"Training for {total_timesteps:,} timesteps...\n")
        start_time = time.time()
        model.learn(total_timesteps=total_timesteps)
        elapsed = time.time() - start_time
        
        model.save("pick_block_ppo_isaac")
        print("\n" + "=" * 70)
        print(f"✓ Training complete! ({elapsed/60:.1f} minutes)")
        print(f"✓ Model saved as 'pick_block_ppo_isaac'")
        print("=" * 70)
        
        env.close()
        return model
    
    except Exception as e:
        print(f"✗ Training error: {e}")
        import traceback
        traceback.print_exc()
        return None

print("✓ train_pick_place_isaac() function defined")

In [ ]:
# Train the model (adjust timesteps as needed)
# model_isaac = train_pick_place_isaac(total_timesteps=100000)

In [ ]:
def evaluate_model_isaac(model_path="pick_block_ppo_isaac", num_episodes=3):
    """Evaluate the Isaac Sim model."""
    from stable_baselines3 import PPO
    
    print("=" * 70)
    print("Evaluating Pick Block Model (Isaac Sim)")
    print("=" * 70)
    
    try:
        model = PPO.load(model_path)
        
        touches = 0
        lifts = 0
        
        for ep in range(num_episodes):
            print(f"\nEpisode {ep + 1}/{num_episodes}:")
            
            env = PickPlaceEnvIsaacSim(headless=True)
            obs, _ = env.reset()
            
            episode_reward = 0.0
            touched = False
            
            while env.step_count < env.max_episode_steps:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, _ = env.step(action)
                episode_reward += reward
                
                # Get distances from observation
                ee_pos = obs[14:17]
                block_pos = obs[17:20]
                gripper_width = obs[23]
                distance = np.linalg.norm(ee_pos - block_pos)
                
                if distance < 0.12 and gripper_width > 0.01:
                    touched = True
                
                if terminated or truncated:
                    break
            
            if touched:
                touches += 1
            
            print(f"  Reward: {episode_reward:.2f}")
            print(f"  Touched: {'✓ YES' if touched else '✗ NO'}")
            
            env.close()
        
        print("\n" + "=" * 70)
        print(f"Touch Success Rate: {touches}/{num_episodes} ({100*touches/num_episodes:.0f}%)")
        print("=" * 70)
    
    except Exception as e:
        print(f"✗ Evaluation error: {e}")
        import traceback
        traceback.print_exc()

print("✓ evaluate_model_isaac() function defined")

## Key Differences: MuJoCo vs Isaac Sim

| Aspect | MuJoCo | Isaac Sim |
|--------|--------|-----------|
| Physics Engine | Native MuJoCo (spring-based) | Omniverse Physics (ODE-based) |
| Model Format | XML (.xml files) | USD (.usd files) |
| Rendering | Offscreen/headless | Real-time Omniverse GUI |
| GPU Support | CPU-only (fast) | GPU-accelerated physics |
| API | Python mujoco library | Isaac SDK Python API |
| Parallelization | Multi-environment CPU | Multi-environment GPU |
| Robot Loading | XML model building | USD prebuilt libraries |
| Integration | Standalone | Omniverse ecosystem |

## Setup Instructions for Isaac Sim

1. **Install NVIDIA Isaac Sim** (if not already installed):
   ```bash
   # Visit: https://developer.nvidia.com/isaac-sim
   # Download and install version 4.0 or later
   ```

2. **Initialize Isaac Sim environment**:
   ```bash
   cd ~/.local/share/ov/pkg/isaac-sim-4.0.0
   source setup_conda_env.sh
   ```

3. **Install Python dependencies**:
   ```bash
   pip install gymnasium stable-baselines3 torch
   ```

4. **Run this notebook**:
   ```bash
   jupyter notebook pick_and_place_train.ipynb
   ```

## Notes

- This notebook uses the Isaac Sim Python API for environment creation
- Training can be significantly faster with GPU acceleration
- Models trained in Isaac Sim are compatible with the MuJoCo version for comparison
- Some features may differ between physics engines (e.g., contact simulation)